# Dark Channel Prior (DCP)
He, Sun, Tang (CVPR 2009 / TPAMI 2011). Classical, training-free: no network, no weights, no GPU.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE     = "/content/drive/MyDrive"
SRC       = f"{DRIVE}/O-HAZE"          # original O-HAZE: hazy/ + GT/
DATA_ROOT = f"{DRIVE}/O-HAZE-1600"     # resized copy, shared by all methods and the evaluation
OUT       = f"{DRIVE}/outputs/dcp" # dehazed images, same file names as hazy/

MAX_IMAGES = 5             # first N pairs (None = all 45)
RESIZE = 1600              # long side; O-HAZE is ~3000x4000

In [ ]:
#resize hazy + GT once (skipped if DATA_ROOT already exists), same resizing as before:
#hazy: long side -> RESIZE (bicubic), GT: resized to the hazy size
import os, glob
from PIL import Image

def resize_long(img, long_side):
    if long_side <= 0 or max(img.size) <= long_side:
        return img
    s = long_side / max(img.size)
    return img.resize((round(img.size[0] * s), round(img.size[1] * s)), Image.BICUBIC)

if not os.path.isdir(f"{DATA_ROOT}/hazy"):
    os.makedirs(f"{DATA_ROOT}/hazy"); os.makedirs(f"{DATA_ROOT}/GT")
    pairs = list(zip(sorted(glob.glob(f"{SRC}/hazy/*")), sorted(glob.glob(f"{SRC}/GT/*"))))[:MAX_IMAGES]
    for h, g in pairs:
        hazy = resize_long(Image.open(h).convert('RGB'), RESIZE)
        gt   = Image.open(g).convert('RGB').resize(hazy.size, Image.BICUBIC)
        stem = lambda p: os.path.splitext(os.path.basename(p))[0]
        hazy.save(f"{DATA_ROOT}/hazy/{stem(h)}.png"); gt.save(f"{DATA_ROOT}/GT/{stem(g)}.png")

hazy_paths = sorted(glob.glob(f"{DATA_ROOT}/hazy/*"))
os.makedirs(OUT, exist_ok=True)
print(len(hazy_paths), "images:", [os.path.basename(p) for p in hazy_paths])

## Parameters

In [ ]:
# DCP settings -- defaults follow He et al. (TPAMI 2011)
PATCH = 15                 # local patch size for the dark channel (min filter)
OMEGA = 0.95               # keep a little haze for distant objects (aerial perspective), 1.0 = remove all
TOP_PERCENT = 0.001        # brightest 0.1% of the dark channel are candidates for the atmospheric light A
T0 = 0.1                   # lower bound on transmission, avoids amplifying noise in dense haze

# transmission refinement (guided filter, replaces soft matting from the paper)
REFINE = True
GF_RADIUS = 60             # guided filter window radius
GF_EPS = 1e-4              # regularization; smaller = follows edges of the guide image more closely

## Method

In [ ]:
def dark_channel(img, patch=PATCH):
    """Min over RGB, then min over a patch x patch neighbourhood (erosion = local min filter)."""
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (patch, patch))
    return cv2.erode(img.min(axis=2), kernel)

def atmospheric_light(img, dark, top=TOP_PERCENT):
    """Among the brightest `top` fraction of dark-channel pixels, take the one with the highest intensity in the input."""
    n = max(int(dark.size * top), 1)
    cand = np.argpartition(dark.ravel(), -n)[-n:]           # indices of the n largest dark-channel values
    flat = img.reshape(-1, 3)
    return flat[cand[flat[cand].sum(axis=1).argmax()]]       # (3,) RGB

def guided_filter(guide, src, r=GF_RADIUS, eps=GF_EPS):
    """Edge-preserving smoothing of `src`, following the edges of `guide` (He et al., 2010)."""
    box = lambda x: cv2.boxFilter(x, -1, (2 * r + 1, 2 * r + 1))
    mean_I, mean_p = box(guide), box(src)
    cov_Ip = box(guide * src) - mean_I * mean_p
    var_I  = box(guide * guide) - mean_I * mean_I
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return box(a) * guide + box(b)

def dehaze_dcp(img):
    """img: float32 RGB in [0, 1]. Returns (J, t, A, dark)."""
    dark = dark_channel(img)
    A = atmospheric_light(img, dark)
    t = 1 - OMEGA * dark_channel(img / A)                    # transmission estimate
    if REFINE:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        t = guided_filter(gray, t.astype(np.float32))
    J = (img - A) / np.maximum(t, T0)[..., None] + A         # invert I = J*t + A*(1-t)
    return np.clip(J, 0, 1), t, A, dark


def dehaze(hazy):
    """hazy: PIL RGB image -> uint8 RGB array."""
    J, t, A, dark = dehaze_dcp(np.asarray(hazy, dtype=np.float32) / 255)
    return (J * 255 + 0.5).astype(np.uint8)

## Preview a single image

In [ ]:
hazy = Image.open(hazy_paths[0]).convert('RGB')
J, t, A, dark = dehaze_dcp(np.asarray(hazy, dtype=np.float32) / 255)

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, im, title, cmap in zip(ax, [hazy, J, dark, t],
                              ['hazy', 'DCP', 'dark channel', f'transmission t (A={np.round(A, 2)})'],
                              [None, None, 'gray', 'jet']):
    a.imshow(im, cmap=cmap); a.set_title(title); a.axis('off')
plt.tight_layout(); plt.show()

## Run over the dataset

In [ ]:
import time
for p in hazy_paths:
    hazy = Image.open(p).convert('RGB')
    t0 = time.time()
    pred = dehaze(hazy)
    dt = time.time() - t0
    Image.fromarray(pred).save(f"{OUT}/{os.path.basename(p)}")
    print(f"{os.path.basename(p)} | {hazy.size[0]}x{hazy.size[1]} | {dt:.1f}s")

## Evaluate
Score the outputs with `dehazing/evaluation` (see `../DEHAZING_EVALUATION.ipynb`):
```python
per_image, summary = evaluate(DATA_ROOT, {"dcp": OUT})
```